In [2]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.SeqIO import FastaIO
import pandas as pd
import os

# Ruta de entrada
genbank_file = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\Zaire_ebolavirus_genome\genome_2_MH481611.2\sequence.gb"

# Ruta de salida
output_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\anotacion_genoma_2"
csv_output = os.path.join(output_folder, "anotacion_genoma.csv")
fasta_output = os.path.join(output_folder, "proteinas_ebola.fasta")
gff_output = os.path.join(output_folder, "sequence.gff3")

# Crear carpeta de salida si no existe
os.makedirs(output_folder, exist_ok=True)

anotaciones = []
proteinas = []
gff_lines = []

for record in SeqIO.parse(genbank_file, "genbank"):
    seqid = record.id
    for feature in record.features:
        if feature.type in ["CDS", "gene", "tRNA", "rRNA"]:
            start = int(feature.location.start) + 1  # GFF usa coordenadas 1-based
            end = int(feature.location.end)
            strand = "+" if feature.location.strand == 1 else "-"
            qualifiers = feature.qualifiers

            gene = qualifiers.get("gene", ["-"])[0]
            locus_tag = qualifiers.get("locus_tag", ["-"])[0]
            product = qualifiers.get("product", ["-"])[0]
            protein_id = qualifiers.get("protein_id", ["-"])[0] if "protein_id" in qualifiers else "-"
            translation = qualifiers.get("translation", [""])[0] if "translation" in qualifiers else ""

            # CSV
            anotaciones.append({
                "Feature": feature.type,
                "Start": start,
                "End": end,
                "Strand": strand,
                "Gene": gene,
                "Locus_tag": locus_tag,
                "Product": product,
                "Protein ID": protein_id,
                "AA sequence": translation
            })

            # FASTA
            if translation != "":
                proteinas.append(SeqRecord(
                    seq=feature.qualifiers["translation"][0],
                    id=protein_id if protein_id != "-" else gene,
                    description=product
                ))

            # GFF3
            attributes = []
            if gene != "-":
                attributes.append(f"Name={gene}")
            if locus_tag != "-":
                attributes.append(f"locus_tag={locus_tag}")
            if product != "-":
                attributes.append(f"product={product}")
            if protein_id != "-":
                attributes.append(f"protein_id={protein_id}")
            attr_str = ";".join(attributes)

            gff_lines.append(f"{seqid}\tGenBank\t{feature.type}\t{start}\t{end}\t.\t{strand}\t.\t{attr_str}")

# Guardar CSV
df = pd.DataFrame(anotaciones)
df.to_csv(csv_output, index=False)
print(f"✅ anotacion_genoma.csv generado en: {csv_output}")

# Guardar FASTA
if proteinas:
    with open(fasta_output, "w") as f:
        fasta_writer = FastaIO.FastaWriter(f, wrap=None)
        fasta_writer.write_file(proteinas)
    print(f"✅ proteinas_ebola.fasta generado en: {fasta_output}")

# Guardar GFF3
with open(gff_output, "w") as gff_file:
    gff_file.write("##gff-version 3\n")
    for line in gff_lines:
        gff_file.write(line + "\n")
print(f"✅ sequence.gff3 generado en: {gff_output}")


✅ anotacion_genoma.csv generado en: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\anotacion_genoma_2\anotacion_genoma.csv
✅ proteinas_ebola.fasta generado en: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\anotacion_genoma_2\proteinas_ebola.fasta
✅ sequence.gff3 generado en: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\anotacion_genoma_2\sequence.gff3


c:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\.venv\Lib\site-packages\Bio\SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
